# Moshi Compression — Session S14 (code-only cache for Phase 1 inputs)

**Purpose.** S2..S13 cached teacher `hidden` (post `out_norm`) + text top-256
but dropped the pre-transformer input. Phase 1 needs the student to reconstruct
that input from Mimi codes + the frozen `emb`/`text_emb` modules. This session
caches just the codes — small (~2 GB) and fast (no teacher forward).

**Carry-over.** Same audio slicing as S2..S13: identical `SHARD_PLAN` and
`FLAC_SKIP` math, so window `i` here produces the codes for the audio segment
that produced `hidden[i]` in the existing cache. `divmod(i, 5000)` still maps
into the 12 memmap parts — and `i` itself indexes this single codes.npy.

**Output.** `kaggle.com/mhassann/moshi-cache-codes`:
* `codes.npy` int16 `(60 000, 17, 375)` — 2.29 GB
* `MANIFEST.md`, `env.txt`, `throughput.json`

**Budget.** Single Kaggle session, ~2.5 h. No teacher load → no sharding →
single GPU. Estimated 500+ windows/min (Mimi is ~350 M vs teacher 7.7 B).


## Cell 1 — Global patches


In [1]:
# Disable torch.compile globally (T4 Inductor emits bf16 intrinsics → crash).
import os
import torch

os.environ["TORCH_COMPILE_DISABLE"] = "1"
torch._dynamo.config.disable = True
print("torch.compile disabled globally")


torch.compile disabled globally


## Cell 2 — Environment verification (single-GPU OK)


In [2]:
import sys
print("python :", sys.version)
print("torch  :", torch.__version__, "  cuda:", torch.version.cuda)
print("device count    :", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} = {p.name}, sm {p.major}.{p.minor}, "
          f"total {p.total_memory / 1e9:.1f} GB")

# S14 only uses cuda:0 for Mimi. Dual-GPU is fine but not required.
assert torch.cuda.device_count() >= 1, "Need at least 1 GPU"
torch.cuda.set_device(0)
print("cuda:0 anchored")
print("=== environment check PASSED ===")


python : 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
torch  : 2.10.0+cu128   cuda: 12.8
device count    : 2
  cuda:0 = Tesla T4, sm 7.5, total 15.6 GB
  cuda:1 = Tesla T4, sm 7.5, total 15.6 GB
cuda:0 anchored
=== environment check PASSED ===


## Cell 3 — Installs (S2 pins, minus bnb which isn't needed here)


In [3]:
import subprocess, sys, os

for pkg in [
    "transformers==4.44.2",
    "accelerate==0.33.0",
    "sentencepiece",
    "einops",
    "soundfile",
    "soxr",       # replaces librosa — scipy/numpy conflict on Kaggle
]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

MOSHI_SRC = "/kaggle/input/datasets/tasfiatanha/moshi-repo/moshi/moshi"
MOSHI_DST = "/kaggle/working/moshi_repo"

if not os.path.exists(MOSHI_DST):
    ret = subprocess.run(["cp", "-r", MOSHI_SRC, MOSHI_DST], capture_output=True, text=True)
    if ret.returncode != 0:
        raise RuntimeError(f"cp failed:\n{ret.stderr}")

ret = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", MOSHI_DST],
    capture_output=True, text=True
)
if ret.returncode != 0:
    print("pip stdout:", ret.stdout)
    print("pip stderr:", ret.stderr)
    raise RuntimeError("moshi editable install failed")

import site, importlib
site.addsitedir(site.getsitepackages()[0])
if MOSHI_DST not in sys.path:
    sys.path.insert(0, MOSHI_DST)
importlib.invalidate_caches()

import moshi, transformers, soundfile, soxr
print(f"moshi from: {moshi.__file__}")
print("=== installs OK ===")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 83.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 86.3 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you 

moshi from: /kaggle/working/moshi_repo/moshi/__init__.py
=== installs OK ===


In [4]:
# Monkey-patch CUDAGraphed to no-op (defensive — Mimi path shouldn't hit it,
# but installing it keeps behaviour parity with S0..S13).
import moshi.utils.compile as _moshi_compile

class _NoGraph:
    def __init__(self, fn, *a, **kw): self.fn = fn
    def __call__(self, *a, **kw):    return self.fn(*a, **kw)

_moshi_compile.CUDAGraphed = _NoGraph
print("CUDAGraphed monkey-patched to no-op")


CUDAGraphed monkey-patched to no-op


## Cell 4 — Load Mimi only on cuda:0

We skip the teacher LM entirely — S14 only needs Mimi.encode. This saves
~7.7 GB of VRAM and ~2 min of load time vs S2's full teacher path.

Weight download uses the same HF mirror as S0..S13; we delete the 14 GB
model.safetensors immediately after Mimi loads (we never touch it).


In [5]:
import torch, pathlib, time, shutil
from moshi.models.loaders import CheckpointInfo
from huggingface_hub import hf_hub_download

REPO_ID  = "kyutai/moshiko-pytorch-bf16"
LOAD_DIR = pathlib.Path("/tmp/moshiko-weights")
LOAD_DIR.mkdir(parents=True, exist_ok=True)

# S14 needs only Mimi + tokenizer. But CheckpointInfo still wants moshi_weights
# to exist, so we download it, load Mimi, and delete it.
MOSHI_FILE = "model.safetensors"
MIMI_FILE  = "tokenizer-e351c8d8-checkpoint125.safetensors"
TOK_FILE   = "tokenizer_spm_32k_3.model"
FILES = {
    MOSHI_FILE : 14_000_000_000,
    MIMI_FILE  : 350_000_000,
    TOK_FILE   : 500_000,
}

def already_done(name, min_sz):
    p = LOAD_DIR / name
    return p.exists() and p.stat().st_size >= min_sz

if not all(already_done(f, s) for f, s in FILES.items()):
    for filename, min_size in FILES.items():
        if already_done(filename, min_size):
            print(f"SKIP {filename}"); continue
        for attempt in range(1, 6):
            print(f"[attempt {attempt}] {filename}")
            try:
                hf_hub_download(repo_id=REPO_ID, filename=filename,
                                local_dir=str(LOAD_DIR), force_download=False)
                if already_done(filename, min_size):
                    print("  OK"); break
            except Exception as e:
                print(f"  Error: {e} — retry in 10s"); time.sleep(10)
        else:
            raise RuntimeError(f"Could not download {filename}")

info = CheckpointInfo(
    moshi_weights = LOAD_DIR / MOSHI_FILE,
    mimi_weights  = LOAD_DIR / MIMI_FILE,
    tokenizer     = LOAD_DIR / TOK_FILE,
    lm_config     = None,
)

print("\nLoading Mimi …")
mimi = info.get_mimi(device="cpu")
mimi = mimi.to(device="cuda:0", dtype=torch.float16)
mimi.eval()
for p in mimi.parameters():
    p.requires_grad_(False)

# Now delete weights — we never load the teacher LM
print("\nDeleting weights from /tmp …")
shutil.rmtree(LOAD_DIR)

import os
stat = os.statvfs("/kaggle/working")
print(f"Disk free: {stat.f_bavail * stat.f_frsize / 1e9:.1f} GB")

# Warm-up: one 30s silent window
with torch.inference_mode():
    silent = torch.zeros(1, 1, 24_000 * 30, dtype=torch.float16, device="cuda:0")
    codes_warm = mimi.encode(silent)
print(f"Mimi output shape on 30s: {codes_warm.shape} dtype={codes_warm.dtype}")
assert codes_warm.shape == (1, 8, 375), codes_warm.shape
assert codes_warm.max().item() < 2048, "Mimi codebook >= 2048 — int16 unsafe"

free, total = torch.cuda.mem_get_info(0)
print(f"cuda:0 free: {free/1e9:.2f} / {total/1e9:.2f} GB")
print("=== Cell 4 PASSED ===")


[attempt 1] model.safetensors


model.safetensors:   0%|          | 0.00/15.4G [00:00<?, ?B/s]

  OK
[attempt 1] tokenizer-e351c8d8-checkpoint125.safetensors


tokenizer-e351c8d8-checkpoint125.safeten(…):   0%|          | 0.00/385M [00:00<?, ?B/s]

  OK
[attempt 1] tokenizer_spm_32k_3.model


tokenizer_spm_32k_3.model:   0%|          | 0.00/553k [00:00<?, ?B/s]

  OK

Loading Mimi …

Deleting weights from /tmp …
Disk free: 20.9 GB
Mimi output shape on 30s: torch.Size([1, 8, 375]) dtype=torch.int64
cuda:0 free: 14.67 / 15.64 GB
=== Cell 4 PASSED ===


## Cell 5 — SHARD_PLAN, audio paths, allocate codes.npy

**Invariant.** The window order here must exactly match S2..S13 so
`codes[i]` lines up with `hidden[i]`. We replicate S2's `SHARD_PLAN` and
`FLAC_SKIP` math verbatim and iterate all 12 (shard, part) tuples in order.

`codes.npy` is pre-allocated at full size — 2.29 GB, comfortably inside the
19.5 GB `/kaggle/working` cap. We open it `w+` for fresh runs or `r+` for
resume (detected by non-zero window at the in-progress offset).


In [6]:
import pathlib, os
import numpy as np

WINDOWS_PER_PART   = 5_000
FLAC_PER_PART      = 12_500
WINDOW_SECONDS     = 30
TARGET_SR          = 24_000
SAMPLES_PER_WINDOW = TARGET_SR * WINDOW_SECONDS  # 720_000
T_FRAMES           = 375
N_CB_TEACHER       = 17         # text + 16 audio slots (Moshi convention)
MIMI_AUDIO_CB      = 8          # Moshiko uses 8 audio codebooks
TEXT_PAD           = 3          # moshi's text pad id
TOTAL_WINDOWS      = 60_000     # = 12 parts × 5 000

# ── Audio source paths (same registry as S2) ─────────────────────────────────
KNOWN_PATHS = {
    "train-clean-100": next(
        (p for p in [
            pathlib.Path("/kaggle/input/datasets/mhassann/librispeech-train-clean-100/LibriSpeech/train-clean-100"),
            pathlib.Path("/kaggle/input/datasets/tasfiatanha/librispeech-train-clean-100/LibriSpeech/train-clean-100"),
            pathlib.Path("/kaggle/input/librispeech-train-clean-100/LibriSpeech/train-clean-100"),
        ] if p.exists()),
        pathlib.Path("/kaggle/input/datasets/mhassann/librispeech-train-clean-100/LibriSpeech/train-clean-100")
    ),
    "train-clean-360": pathlib.Path("/kaggle/input/datasets/manancodes/librispeech-train-clean-360/LibriSpeech/train-clean-360"),
    "train-other-500": pathlib.Path("/kaggle/input/datasets/fredrelec/train-other-500/LibriSpeech/train-other-500"),
}
for name, path in KNOWN_PATHS.items():
    print(f"  {'OK' if path.exists() else 'MISSING'}  {name}  {path}")

def shard_plan_for(shard_idx, part_idx):
    flac_skip = part_idx * FLAC_PER_PART
    take     = FLAC_PER_PART + 3_000
    if shard_idx == 0:
        return [("train-clean-360", flac_skip, take)]
    if shard_idx == 1:
        return [("train-clean-360", 52_000 + flac_skip, take),
                ("train-other-500", max(0, flac_skip - 52_000), take)]
    if shard_idx == 2:
        return [("train-other-500", 30_000 + flac_skip, take)]
    raise ValueError(shard_idx)

# ── Allocate output memmap ───────────────────────────────────────────────────
OUT_DIR   = pathlib.Path("/kaggle/working/cache_codes")
OUT_DIR.mkdir(parents=True, exist_ok=True)
CODES_PATH = OUT_DIR / "codes.npy"

def count_resume(path):
    if not path.exists():
        return 0
    mm = np.memmap(path, dtype="int16", mode="r",
                   shape=(TOTAL_WINDOWS, N_CB_TEACHER, T_FRAMES))
    # A written window has non-default values in the audio rows (1..8).
    # Rows 0 and 9..16 are TEXT_PAD=3 by design; zero-init means "not written".
    # Count sequentially — resume is contiguous.
    done = 0
    for i in range(TOTAL_WINDOWS):
        if int(mm[i, 1, 0]) == 0 and int(mm[i, 1, -1]) == 0 and int(mm[i, 2, 0]) == 0:
            break
        done += 1
    del mm
    return done

already_done = count_resume(CODES_PATH)
print(f"\nAlready written: {already_done}/{TOTAL_WINDOWS} windows")

mode = "r+" if CODES_PATH.exists() else "w+"
codes_mm = np.memmap(CODES_PATH, dtype="int16", mode=mode,
                     shape=(TOTAL_WINDOWS, N_CB_TEACHER, T_FRAMES))

if mode == "w+":
    # Initialise whole array to TEXT_PAD so rows 0 and 9..16 never need writes.
    codes_mm[:] = TEXT_PAD
    codes_mm.flush()
    print("Allocated and initialised codes.npy to TEXT_PAD")

stat = os.statvfs("/kaggle/working")
print(f"/kaggle/working free: {stat.f_bavail * stat.f_frsize / 1e9:.1f} GB")
print(f"codes.npy size: {CODES_PATH.stat().st_size/1e9:.2f} GB")
print("=== Cell 5 PASSED ===")


  OK  train-clean-100  /kaggle/input/datasets/tasfiatanha/librispeech-train-clean-100/LibriSpeech/train-clean-100
  OK  train-clean-360  /kaggle/input/datasets/manancodes/librispeech-train-clean-360/LibriSpeech/train-clean-360
  OK  train-other-500  /kaggle/input/datasets/fredrelec/train-other-500/LibriSpeech/train-other-500

Already written: 0/60000 windows
Allocated and initialised codes.npy to TEXT_PAD
/kaggle/working free: 20.2 GB
codes.npy size: 0.77 GB
=== Cell 5 PASSED ===


## Cell 6 — Cache loop over all 60 k windows

Iterate the 12 `(shard, part)` tuples in order, applying the same audio-buffer
windowing as S2 within each part. For each 30 s window, write Mimi codes into
rows 1..8 of `codes_mm[global_i]`; rows 0 and 9..16 are already TEXT_PAD.

`global_i` is the absolute window index across all 60 k — same as the index
into `hidden.npy` parts via `part_i, local_i = divmod(global_i, 5000)`.

Resume semantics: skip windows with `global_i < already_done`.


In [7]:
import time, json
import numpy as np
import soundfile as sf
import soxr
import torch
import gc

log_rows    = []
t_loop_start = time.time()
n_done      = already_done
n_failed    = 0

def tick(global_i, local_i_in_part, shard_idx, part_idx, t0, local_n):
    elapsed = time.time() - t0
    wpm = (n_done - already_done) / max(elapsed, 1e-6) * 60
    eta_min = (TOTAL_WINDOWS - n_done) / max(wpm, 1e-6) * 60 / 60
    free, _ = torch.cuda.mem_get_info(0)
    row = {
        "global_i":   global_i,
        "shard":      shard_idx,
        "part":       part_idx,
        "local_idx":  local_i_in_part,
        "n_done":     n_done,
        "elapsed_s":  round(elapsed, 1),
        "wpm":        round(wpm, 2),
        "eta_min":    round(eta_min, 1),
        "gpu0_free":  round(free / 1e9, 2),
        "flac_idx":   local_n,
    }
    log_rows.append(row)
    print(f"  [{global_i+1:5d}/{TOTAL_WINDOWS}] shard{shard_idx}p{part_idx} "
          f"local={local_i_in_part:4d}  wpm={wpm:6.1f}  eta={eta_min:5.1f}min  "
          f"gpu0_free={row['gpu0_free']:.1f}")

for shard_idx in (0, 1, 2):
    for part_idx in (0, 1, 2, 3):
        global_offset = (shard_idx * 4 + part_idx) * WINDOWS_PER_PART
        if global_offset + WINDOWS_PER_PART <= already_done:
            print(f"SKIP shard{shard_idx}p{part_idx} (entirely done)")
            continue

        print(f"\n=== shard{shard_idx}p{part_idx}  "
              f"global {global_offset}..{global_offset + WINDOWS_PER_PART - 1} ===")

        local_i = 0
        buf = np.zeros(0, dtype=np.float32)

        for source_name, skip, take in shard_plan_for(shard_idx, part_idx):
            if local_i >= WINDOWS_PER_PART:
                break
            path = KNOWN_PATHS.get(source_name)
            if not (path and path.exists()):
                print(f"  MISSING source: {source_name}")
                continue

            flac_files = sorted(path.rglob("*.flac"))[skip: skip + take]
            print(f"  {source_name}: {len(flac_files)} files from skip={skip}")

            local_n = 0
            for fpath in flac_files:
                if local_i >= WINDOWS_PER_PART:
                    break
                local_n += 1
                try:
                    arr, sr = sf.read(str(fpath), dtype="float32")
                    if arr.ndim > 1:
                        arr = arr.mean(axis=1).astype(np.float32)
                    if sr != TARGET_SR:
                        arr = soxr.resample(arr, sr, TARGET_SR, quality="HQ").astype(np.float32)
                    buf = np.concatenate([buf, arr])
                except Exception as e:
                    print(f"    skip {fpath.name}: {e}")
                    continue

                while buf.shape[0] >= SAMPLES_PER_WINDOW and local_i < WINDOWS_PER_PART:
                    wav_np = buf[:SAMPLES_PER_WINDOW]
                    buf    = buf[SAMPLES_PER_WINDOW:]
                    global_i = global_offset + local_i

                    if global_i < already_done:
                        local_i += 1
                        continue

                    try:
                        wav_t = torch.from_numpy(wav_np.copy()).to(
                            device="cuda:0", dtype=torch.float16
                        ).unsqueeze(0).unsqueeze(0)
                        with torch.inference_mode():
                            codes = mimi.encode(wav_t)  # [1, 8, 375] int64
                        # Write into rows 1..8 of the pre-padded global slot.
                        codes_mm[global_i, 1:1 + MIMI_AUDIO_CB, :] = \
                            codes.squeeze(0).to("cpu", dtype=torch.int16).numpy()
                        n_done += 1

                        if n_done % 200 == 0:
                            codes_mm.flush()

                    except RuntimeError as e:
                        n_failed += 1
                        # Keep slot as TEXT_PAD rows (no Mimi codes) — Phase 1
                        # loader will see the zero audio rows and can skip.
                        codes_mm[global_i, 1:1 + MIMI_AUDIO_CB, :] = 0
                        print(f"    window {global_i} FAILED: {e}")
                        torch.cuda.empty_cache()

                    local_i += 1

                    if (local_i % 500 == 0) or local_i == WINDOWS_PER_PART:
                        tick(global_i, local_i, shard_idx, part_idx, t_loop_start, local_n)

        print(f"  shard{shard_idx}p{part_idx} done: local_i={local_i}/{WINDOWS_PER_PART}")
        codes_mm.flush()
        gc.collect()

codes_mm.flush()
elapsed = time.time() - t_loop_start
final_wpm = (n_done - already_done) / max(elapsed, 1e-6) * 60

# Audit: any global slot whose audio rows are still 0 means a failed encode.
failed_indices = []
for i in range(TOTAL_WINDOWS):
    if int(codes_mm[i, 1, 0]) == 0 and int(codes_mm[i, 1, -1]) == 0 \
       and int(codes_mm[i, 2, 0]) == 0:
        failed_indices.append(i)

log = {
    "total_windows":      TOTAL_WINDOWS,
    "n_done":             int(n_done),
    "n_failed_encode":    int(n_failed),
    "n_zero_at_audit":    len(failed_indices),
    "failed_indices":     failed_indices[:200],   # first 200 only
    "wall_seconds":       round(elapsed, 1),
    "windows_per_minute": round(final_wpm, 2),
    "log":                log_rows,
}
(OUT_DIR / "throughput.json").write_text(json.dumps(log, indent=2))

print(f"\nDone: {n_done}/{TOTAL_WINDOWS} windows in {elapsed/60:.1f} min "
      f"({final_wpm:.1f} w/min)")
print(f"Failed during encode: {n_failed}")
print(f"Zero audio rows at audit: {len(failed_indices)}  "
      f"(first 10: {failed_indices[:10]})")
print("=== Cell 6 PASSED ===")
k


=== shard0p0  global 0..4999 ===
  train-clean-360: 15500 files from skip=0
  [  500/60000] shard0p0 local= 500  wpm= 138.6  eta=429.2min  gpu0_free=14.7
  [ 1000/60000] shard0p0 local=1000  wpm= 238.2  eta=247.7min  gpu0_free=14.7
  [ 1500/60000] shard0p0 local=1500  wpm= 312.9  eta=186.9min  gpu0_free=14.7
  [ 2000/60000] shard0p0 local=2000  wpm= 371.0  eta=156.3min  gpu0_free=14.7
  [ 2500/60000] shard0p0 local=2500  wpm= 417.8  eta=137.6min  gpu0_free=14.7
  [ 3000/60000] shard0p0 local=3000  wpm= 450.2  eta=126.6min  gpu0_free=14.7
  [ 3500/60000] shard0p0 local=3500  wpm= 478.1  eta=118.2min  gpu0_free=14.7
  [ 4000/60000] shard0p0 local=4000  wpm= 504.7  eta=111.0min  gpu0_free=14.7
  [ 4500/60000] shard0p0 local=4500  wpm= 527.6  eta=105.2min  gpu0_free=14.7
  [ 5000/60000] shard0p0 local=5000  wpm= 546.5  eta=100.6min  gpu0_free=14.7
  shard0p0 done: local_i=5000/5000

=== shard0p1  global 5000..9999 ===
  train-clean-360: 15500 files from skip=12500
  [ 5500/60000] shard0p1

## Cell 7 — MANIFEST + env.txt


In [8]:
import json, subprocess, pathlib, torch, transformers

OUT_DIR    = pathlib.Path("/kaggle/working/cache_codes")
CODES_PATH = OUT_DIR / "codes.npy"

throughput = json.loads((OUT_DIR / "throughput.json").read_text())

env_out  = subprocess.run(["pip", "freeze"],    capture_output=True, text=True).stdout
nvid_out = subprocess.run(["nvidia-smi", "-q"], capture_output=True, text=True).stdout
(OUT_DIR / "env.txt").write_text(env_out + "\n=== nvidia-smi ===\n" + nvid_out)

(OUT_DIR / "MANIFEST.md").write_text(f"""# MANIFEST — moshi-cache-codes

Phase-1 input codes for all 60 000 cached windows.

| Key | Value |
|---|---|
| n_windows         | {throughput['total_windows']} |
| n_done            | {throughput['n_done']} |
| n_failed_encode   | {throughput['n_failed_encode']} |
| n_zero_at_audit   | {throughput['n_zero_at_audit']} |
| windows_per_minute| {throughput['windows_per_minute']} |
| wall_minutes      | {throughput['wall_seconds']/60:.1f} |
| codes_size_gb     | {CODES_PATH.stat().st_size/1e9:.2f} |
| torch             | {torch.__version__} |
| transformers      | {transformers.__version__} |

## File layout
```
codes.npy   int16   (60000, 17, 375)   ≈ 2.29 GB
```

Row layout per window:
* `codes[i, 0, :]`     — text stream, TEXT_PAD=3 (unused in Phase 1)
* `codes[i, 1:9, :]`   — Mimi 8 audio codebooks (the real signal)
* `codes[i, 9:17, :]`  — TEXT_PAD=3 (unused acoustic slots, Moshiko uses 8)

## Alignment with Phase-0 cache
Window order here is identical to S2..S13. For any `i in [0, 60000)`:
```
part_i, local_i = divmod(i, 5000)          # maps into one of 12 hidden parts
```
so `codes[i]` encodes the same 30 s segment that produced `hidden[i]`.

## Phase-1 load pattern
```python
import numpy as np
codes = np.memmap("/kaggle/input/moshi-cache-codes/codes.npy",
                  dtype="int16", mode="r", shape=(60_000, 17, 375))
window_codes = torch.from_numpy(codes[i].astype(np.int64))   # [17, 375]
```

Feed into teacher's frozen `emb`/`text_emb` stack to recover the
pre-transformer input sum that the student's `in_adapter` consumes.

## Failed indices
{throughput['n_zero_at_audit']} windows failed Mimi encode (all-zero rows).
Phase-1 dataloader should either skip these or treat them as a no-op batch.
First 10: {throughput['failed_indices'][:10]}
""")

print("Files in OUT_DIR:")
for p in sorted(OUT_DIR.iterdir()):
    if p.is_file():
        print(f"  {p.name:<20} {p.stat().st_size/1e6:8.1f} MB")
print("=== Cell 7 PASSED ===")


Files in OUT_DIR:
  MANIFEST.md               0.0 MB
  codes.npy               765.0 MB
  env.txt                   0.0 MB
  throughput.json           0.0 MB
=== Cell 7 PASSED ===


## Cell 8 — Push `moshi-cache-codes`


In [9]:
import subprocess, json, pathlib, os

OUT_DIR  = pathlib.Path("/kaggle/working/cache_codes")
username = os.environ.get("KAGGLE_USERNAME", "mhassann")

dataset_id = f"{username}/moshi-cache-codes"
metadata = {
    "title":    "moshi-cache-codes",
    "id":       dataset_id,
    "licenses": [{"name": "CC0-1.0"}],
}
(OUT_DIR / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2))
print(f"Dataset id: {dataset_id}")

total_gb = 0.0
for p in sorted(OUT_DIR.iterdir()):
    if p.is_file():
        gb = p.stat().st_size / 1e9
        total_gb += gb
        print(f"  {p.name:<30} {gb*1000:8.1f} MB")
print(f"  {'TOTAL':<30} {total_gb:8.2f} GB")
assert total_gb < 20, f"Upload {total_gb:.1f} GB larger than expected — investigate"

print("\nkaggle datasets create …")
r = subprocess.run(
    ["kaggle", "datasets", "create", "-p", str(OUT_DIR)],
    capture_output=True, text=True,
)
print(r.stdout or "(no stdout)")

if r.returncode == 0:
    print(f"SUCCESS — kaggle.com/{dataset_id}")
else:
    print(f"Create failed (rc={r.returncode}) — trying version bump")
    r2 = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(OUT_DIR),
         "-m", "S14 codes cache"],
        capture_output=True, text=True,
    )
    print(r2.stdout or "(no stdout)")
    if r2.returncode != 0:
        print("STDERR:", r2.stderr)
    else:
        print(f"SUCCESS — kaggle.com/{dataset_id}")

print("\n=== S14 COMPLETE ===")
print("Next: Phase-1 training notebook.")


Dataset id: mhassann/moshi-cache-codes
  MANIFEST.md                         0.0 MB
  codes.npy                         765.0 MB
  dataset-metadata.json               0.0 MB
  env.txt                             0.0 MB
  throughput.json                     0.0 MB
  TOTAL                              0.77 GB

kaggle datasets create …
Starting upload for file throughput.json
Upload successful: throughput.json (28KB)
Starting upload for file codes.npy
Upload successful: codes.npy (730MB)
Starting upload for file MANIFEST.md
Upload successful: MANIFEST.md (1KB)
Starting upload for file env.txt
Upload successful: env.txt (38KB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/mhassann/moshi-cache-codes

SUCCESS — kaggle.com/mhassann/moshi-cache-codes

=== S14 COMPLETE ===
Next: Phase-1 training notebook.
